In [0]:
from pyspark.sql import SparkSession

spark.sql("CREATE DATABASE IF NOT EXISTS flight_data_gold")
spark.sql("USE flight_data_gold")

DataFrame[]

In [0]:
%sql
CREATE TABLE IF NOT EXISTS flight_data_gold.factflights(
  flight_id string,
  date_key DATE,
  airline_key STRING,
  distance INT,
  origin_airport STRING,
  destination_airport STRING,
  scheduled_departure STRING,
  departure_time STRING,
  departure_delay INT,
  scheduled_time INT,
  elapsed_time INT,
  air_time INT,
  scheduled_arrival STRING,
  arrival_time STRING,
  arrival_delay INT,
  air_system_delay INT,
  security_delay INT,
  airline_delay INT,
  late_aircraft_delay INT,
  weather_delay INT,
  total_delay INT,
  diverted INT,
  cancelled INT,
  cancellation_reason STRING
);

In [0]:
%sql

WITH temp_facts AS(
  SELECT 
    dimfl.flight_id,
    dimd.date_key,
    diml.airline_id as airline_key,
    dimfl.distance,
    dimp1.airport_id as origin_airport,
    dimp2.airport_id as destination_airport,
    dimfd.scheduled_departure,
    dimfd.departure_time,
    dimfd.departure_delay,
    dimfd.scheduled_time,
    dimfd.elapsed_time,
    dimfd.air_time,
    dimfd.scheduled_arrival,
    dimfd.arrival_time,
    dimfd.arrival_delay,
    dimfd.air_system_delay,
    dimfd.security_delay,
    dimfd.airline_delay,
    dimfd.late_aircraft_delay,
    dimfd.weather_delay,
    (dimfd.air_system_delay + dimfd.security_delay + dimfd.airline_delay + dimfd.late_aircraft_delay + dimfd.weather_delay) AS total_delay,
    dimfl.diverted,
    dimfl.cancelled,
    dimfl.cancellation_reason
  FROM flight_data_silver.dimflightlist dimfl
  JOIN flight_data_silver.dimflightdetails dimfd
    ON dimfl.flight_id = dimfd.flight_id
  JOIN flight_data_silver.dimdates dimd
    ON dimfl.date = dimd.date_key
  JOIN flight_data_silver.dimairlines diml
    ON dimfl.airline = diml.airline_id
  JOIN flight_data_silver.dimairports dimp1
    ON dimfl.origin_airport = dimp1.airport_id
  JOIN flight_data_silver.dimairports dimp2
    ON dimfl.destination_airport = dimp2.airport_id
)

MERGE INTO flight_data_gold.factflights AS ff
USING temp_facts AS tf
ON ff.flight_id = tf.flight_id

WHEN MATCHED AND (
  ff.date_key <> tf.date_key OR
  ff.airline_key <> tf.airline_key OR
  ff.origin_airport <> tf.origin_airport OR
  ff.destination_airport <> tf.destination_airport
) THEN
  UPDATE SET
  ff.date_key = tf.date_key,
  ff.airline_key = tf.airline_key,
  ff.origin_airport = tf.origin_airport,
  ff.destination_airport = tf.destination_airport

WHEN NOT MATCHED THEN
  INSERT (flight_id, date_key, airline_key, distance, origin_airport, destination_airport, scheduled_departure,  departure_time, departure_delay, scheduled_time, elapsed_time, air_time, scheduled_arrival, arrival_time, arrival_delay, air_system_delay, security_delay, airline_delay, late_aircraft_delay, weather_delay, total_delay, diverted, cancelled, cancellation_reason)
  VALUES (tf.flight_id, tf.date_key, tf.airline_key, tf.distance, tf.origin_airport, tf.destination_airport, tf.scheduled_departure, tf.departure_time, tf.departure_delay, tf.scheduled_time, tf.elapsed_time, tf.air_time, tf.scheduled_arrival, tf.arrival_time, tf.arrival_delay, tf.air_system_delay, tf.security_delay, tf.airline_delay, tf.late_aircraft_delay, tf.weather_delay, tf.total_delay, tf.diverted, tf.cancelled, tf.cancellation_reason)

In [0]:
%sql

SELECT * FROM flight_data_gold.factflights

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:470)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data